# Step 3 — Raw data-quality audit of the anime and user tables

**Purpose.** Step 1 compared the raw data with the author's description. This stage looks inside the columns that the research questions will depend on, and fixes the `null` username problem found in step 1.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1–3 only read the two small tables and take seconds. Part 4 reads two columns of the 80M-row list table and takes a few minutes; set `RUN_LISTS = False` in the setup cell to skip it.

**Reading the output.** Each part ends with an *Observations* cell marked TODO. Fill it in after the run, writing what you see before interpreting it.

| Part | Question |
|---|---|
| 1 | Which real values does pandas silently turn into `NaN`? |
| 2 | Anime table: placeholders, impossible values, how reliable is `score`, what can a pre-airing model use? |
| 3 | User table: placeholders, impossible ages and dates, suspicious aggregates |
| 4 | Do the per-user counts in `UserList` agree with the rows in `UserAnimeList`? |
| 5 | The users with no row in the list table: do they have other values, which would mean the list table is missing them? |

**Where this step sits.** Step 3 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). This notebook only **finds and counts** problems. It does not clean anything and it does not compare columns statistically. Cleaning rules are decided in step 4, from the evidence found here.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 4, which reads 80M rows

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

## 1. Real values that pandas turns into `NaN`

By default pandas reads strings such as `null`, `NA` or `None` as missing. In step 1 this made the username `null` disappear. Here every cell of the two small tables is read as raw text, and the cells that pandas would have turned into `NaN` are counted per column. Both tables are then loaded again with only the empty string treated as missing.

In [ ]:
NA_TOKENS = {"null", "NULL", "NA", "N/A", "n/a", "NaN", "nan", "None", "#N/A", "#NA",
             "-NaN", "-nan", "<NA>", "1.#IND", "1.#QNAN", "-1.#IND", "-1.#QNAN", "#N/A N/A"}

for file_name in ["AnimeList.csv", "UserList.csv"]:
    text = pd.read_csv(paths[file_name], dtype=str, keep_default_na=False)
    hits = {col: int(text[col].isin(NA_TOKENS).sum()) for col in text.columns}
    hits = {col: count for col, count in hits.items() if count}
    print(file_name, "-> columns with hidden NA tokens:", hits or "none")
    for col in hits:
        print("   ", col, text.loc[text[col].isin(NA_TOKENS), col].value_counts().head(5).to_dict())
    del text

# Safe load: only the empty string is missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("\nanime:", anime.shape, "| users:", users.shape)
print("username NaN:", users.username.isna().sum(),
      "| username unique:", users.username.is_unique,
      "| distinct usernames:", f"{users.username.nunique():,}", "(description: 302,675)")
assert users.username.is_unique

### Observations for part 1

**TODO (fill in after the run):** which columns lost real values, how many, and which values. Does `users` now have 302,675 distinct usernames?

## 2. Anime table

Columns that matter for the candidate questions: `type`, `source`, `episodes`, `duration`, `aired_string`, `genre`, `status`, `rating` (age rating), `related` (sequels, prequels), and the audience columns `score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`.

For a model that predicts the score **before an anime airs**, the audience columns (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) are only known after airing, so they must be excluded as leakage.

In [ ]:
def missing_overview(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }).sort_values("missing_pct", ascending=False)


print(anime.dtypes.value_counts().to_dict())
print(missing_overview(anime).head(12))

# Placeholder values that are not NaN but also carry no information
PLACEHOLDERS = {"", "Unknown", "Not available", "None", "?", "0000-00-00"}
text_cols = anime.select_dtypes(exclude="number").columns
placeholder_counts = pd.Series({c: int(anime[c].isin(PLACEHOLDERS).sum()) for c in text_cols})
print("\nplaceholder values per column:")
print(placeholder_counts[placeholder_counts > 0].sort_values(ascending=False))

print("\nduplicated anime_id:", anime.anime_id.duplicated().sum())

In [ ]:
# Numeric columns: are the ranges plausible?
num_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes"]
print(anime[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

# How reliable is the published score?
print("\nscore == 0:", (anime.score == 0).sum(),
      "| scored_by == 0:", (anime.scored_by == 0).sum(),
      "| both:", ((anime.score == 0) & (anime.scored_by == 0)).sum())
for k in (10, 50, 100, 1000):
    share = (anime.scored_by < k).mean()
    print(f"anime with scored_by < {k:>4}: {(anime.scored_by < k).sum():>6,} ({share:.1%})")
print("episodes == 0:", (anime.episodes == 0).sum())

In [ ]:
# Categorical columns
for col in ["type", "source", "status", "rating"]:
    print(f"\n{col}: {anime[col].nunique(dropna=False)} distinct values")
    print(anime[col].value_counts(dropna=False).head(12))

In [ ]:
# Release year from aired_string (the first 4-digit number is the start year)
aired_year = pd.to_numeric(anime["aired_string"].str.extract(r"(\d{4})", expand=False), errors="coerce")
print("aired_string without a year:", aired_year.isna().sum(), f"({aired_year.isna().mean():.1%})")
print(aired_year.describe().round(1))
print(anime.loc[aired_year.isna(), "aired_string"].value_counts().head(5))
print("examples:", anime["aired_string"].dropna().head(5).tolist())

# Episode length in minutes from the text in `duration`
print("\n", anime["duration"].value_counts().head(8))
hours = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*hr")[0], errors="coerce").fillna(0)
minutes = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*min")[0], errors="coerce").fillna(0)
has_time = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
duration_min = (hours * 60 + minutes).where(has_time)
print("duration not parsable:", (~has_time).sum(), f"({(~has_time).mean():.1%})")
print(duration_min.describe(percentiles=[.01, .5, .99]).round(1))

In [ ]:
# genre holds several values in one cell
genres = anime["genre"].dropna().str.split(", ")
print("anime without genre:", anime["genre"].isna().sum())
print("genres per anime:", genres.str.len().describe().round(2).to_dict())
print("distinct genres:", genres.explode().nunique())
print(genres.explode().value_counts().head(10))

# related: how are sequels stored, and how many anime have one?
print("\nexample:", anime["related"].dropna().head(2).tolist())
for relation in ["Sequel", "Prequel", "Side story", "Spin-off"]:
    share = anime["related"].str.contains(f"'{relation}'", na=False).mean()
    print(f"anime with a '{relation}' entry: {share:.1%}")

### Observations for part 2

**TODO (fill in after the run):**
- Placeholders and missingness: which columns are unusable or need a placeholder-to-`NaN` step?
- Implausible values: `episodes == 0`, extreme `duration`, release years outside 1900–2018.
- Reliability of `score`: how many anime have very few raters?
- Whether `related` is parsable enough for the sequel question, and whether `aired_string` and `duration` can be parsed for the time-trend and ML questions.

## 3. User table

In [ ]:
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
stat_cols = ["user_days_spent_watching", "stats_mean_score", "stats_rewatched", "stats_episodes"]

print(users.dtypes.value_counts().to_dict())
print(missing_overview(users).head(8))
print("access_rank non-missing:", users["access_rank"].notna().sum())

print("\ngender:")
print(users["gender"].value_counts(dropna=False))
print("\ndistinct locations:", users["location"].nunique(),
      "| missing:", users["location"].isna().sum())
print(users["location"].value_counts().head(10))

In [ ]:
# Dates: placeholders and implausible values
for col in ["birth_date", "join_date", "last_online"]:
    parsed = pd.to_datetime(users[col], errors="coerce")
    print(f"{col:12s} unparsable/missing: {parsed.isna().sum():>7,}",
          "| '0000-00-00':", int((users[col] == "0000-00-00").sum()),
          "| earliest:", parsed.min(), "| latest:", parsed.max())
    users[col + "_dt"] = parsed

age_2018 = (pd.Timestamp("2018-01-01") - users["birth_date_dt"]).dt.days / 365.25
print("\nage at 2018-01-01:")
print(age_2018.describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
print("age < 8:", int((age_2018 < 8).sum()), "| age > 80:", int((age_2018 > 80).sum()))
print("last_online before 2000:", int((users["last_online_dt"].dt.year < 2000).sum()),
      "| joined after last_online:", int((users["join_date_dt"] > users["last_online_dt"]).sum()))

# Aggregates reported per user
print(users[stat_cols].describe(percentiles=[.01, .5, .99, .999]).T.round(2))
print("stats_mean_score == 0:", int((users["stats_mean_score"] == 0).sum()))
print("users declaring no anime at all:", int((users[count_cols].sum(axis=1) == 0).sum()))

### Observations for part 3

**TODO (fill in after the run):** which gender and location values need grouping or cleaning, how many ages are impossible, whether `last_online` and `join_date` look reliable, and whether any aggregate has extreme values (candidates for truncation or robust statistics).

## 4. Do the per-user counts agree with the list table?

`UserList` reports how many anime each user has in each status. `UserAnimeList` holds the actual rows. If both come from the same crawl, the counts should match. This also settles the 9 users in step 1 who declare anime but have no rows.

In [ ]:
if RUN_LISTS:
    lists = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "my_status"],
        dtype={"username": "category", "my_status": "int8"}, **na_kwargs,
    )
    print(f"list rows: {len(lists):,} | username NaN: {lists.username.isna().sum()}")

    per_status = lists.groupby(["username", "my_status"], observed=True).size().unstack(fill_value=0)
    print("usernames in lists but not in users:", len(set(per_status.index) - set(users.username)))

    status_to_col = {1: "user_watching", 2: "user_completed", 3: "user_onhold",
                     4: "user_dropped", 6: "user_plantowatch"}
    actual = per_status.reindex(columns=list(status_to_col)).fillna(0).astype(int)
    actual.columns = [c + "_actual" for c in status_to_col.values()]
    actual.index = actual.index.astype(object)
    declared = users.set_index("username")[count_cols].add_suffix("_declared")
    merged = declared.join(actual, how="left").fillna(0)
    print("merge rows:", len(merged), "| users:", len(users))

    for col in count_cols:
        diff = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} exact match: {(diff == 0).mean():6.1%} | median diff: {diff.median():.0f}"
              f" | mean abs diff: {diff.abs().mean():.2f}")

    declared_total = merged[[c + "_declared" for c in count_cols]].sum(axis=1)
    actual_total = merged[[c + "_actual" for c in count_cols]].sum(axis=1)
    print("\ndeclare 0 but have rows:", int(((declared_total == 0) & (actual_total > 0)).sum()),
          "| declare > 0 but have no rows:", int(((declared_total > 0) & (actual_total == 0)).sum()))
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 4

**TODO (fill in after the run):** how often do declared and actual counts agree, in which direction do they differ, and how many users declare anime but have no rows? If the counts disagree widely, `UserList` aggregates should not be used as ground truth for the questions.

## 5. The users with no row in the list table: is the list table missing them?

In step 1 we found **19,631 users** who exist in `UserList` but own no row in `UserAnimeList`. For 19,622 of them the profile declares 0 anime, so an empty list is the natural explanation. But a declared count is only one column. If these users also have **other activity values** (days spent watching, episodes watched, a mean score), their list may exist on MyAnimeList and simply be missing from our list table.

| What we see among these users | What it means |
|---|---|
| `user_days_spent_watching`, `stats_episodes` and `stats_mean_score` are all 0 | The list is really empty. Nothing is missing |
| Many have positive values there | They watched and rated anime, so their list is **missing from the list table** (not crawled, private, or lost) |
| Their profile fields (gender, birth date, location) are filled as often as for other users | They are normal active profiles. If those fields are mostly empty, they are probably inactive accounts |

In [ ]:
if RUN_LISTS:
    users["has_rows"] = users["username"].isin(actual.index)
    no_rows = users[~users["has_rows"]]
    with_rows = users[users["has_rows"]]
    declared_size = no_rows[count_cols].sum(axis=1)
    print(f"users without a row in the list table: {len(no_rows):,} | with rows: {len(with_rows):,}")
    print(f"  declared list size 0: {int((declared_size == 0).sum()):,} | declared list size > 0: {int((declared_size > 0).sum()):,}")

    print("\nactivity values of the users without rows (an empty account would show 0 everywhere):")
    for col in ["user_days_spent_watching", "stats_episodes", "stats_rewatched", "stats_mean_score"]:
        s = no_rows[col]
        print(f"  {col:26s} missing: {int(s.isna().sum()):>6,} | zero: {int((s == 0).sum()):>6,} | positive: {int((s > 0).sum()):>6,}"
              f" | max: {s.max():,.1f}")
    active = ((no_rows["user_days_spent_watching"] > 0) | (no_rows["stats_episodes"] > 0) | (no_rows["stats_mean_score"] > 0))
    print(f"\nusers without rows that show activity in at least one of the three columns: {int(active.sum()):,}")

    print("\nprofile fields filled (share of users), compared with the users that have rows:")
    fields = ["gender", "birth_date", "location", "join_date", "last_online"]
    print(pd.DataFrame({"no_rows": no_rows[fields].notna().mean(),
                        "with_rows": with_rows[fields].notna().mean()}).round(3).to_string())

    print("\nusers without rows that declare anime (the unexplained group):")
    cols_show = ["username"] + count_cols + ["user_days_spent_watching", "stats_episodes", "stats_mean_score"]
    print(no_rows.loc[declared_size > 0, cols_show].sort_values("user_completed", ascending=False).head(15).to_string(index=False))
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 5

**TODO (fill in after the run):** How many of the 19,631 users show activity (days, episodes or a mean score above 0)? Are the 19,622 users with a declared size of 0 also 0 in the three activity columns? Are their profile fields filled as often as for the users with rows? What do the 9 users who declare anime look like? Decide: are these users empty accounts (drop them from list analyses) or users whose list is missing (report how many)?